# Download the TCIA datasets to Google Drive

Downloads the four TCIA datasets with IBM's `ascli` tool and saves each one to
`MyDrive/BrainMRI_Data/raw/<cohort>/` as **one `.tar` archive** plus its clinical/metadata files.

| Key | Dataset | Size | Needed by |
| --- | --- | --- | --- |
| `rhuh` | RHUH-GBM | 2.9 GB | M3 (10 Oct) |
| `mu` | MU-Glioma-Post | 11 GB | M3 (10 Oct) |
| `africa` | BraTS-Africa | 1.6 GB | Final tests (21 Oct) |
| `utsw` | UTSW-Glioma | 22.9 GB | Final tests (21 Oct) |

**How to run:** CPU runtime is fine (no GPU needed). Choose the datasets in step 1, then **Runtime → Run all**.
- Step 2 installs Ruby and `ascli` (about 10–15 minutes, once per session).
- TCIA warns that `ascli` is slower on free Colab. If a download is very slow, use the browser + IBM Aspera Connect route on your PC instead (see `docs/DATA_DOWNLOAD.md`).
- The datasets are done one at a time; each one is deleted from the Colab disk after it is safely on Drive, so even UTSW fits.

In [ ]:
# 1. Choose the datasets and connect Drive
COHORTS = ["rhuh", "mu"]          # add "africa", "utsw" when you want them (needed by 21 Oct)

import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE_RAW = "/content/drive/MyDrive/BrainMRI_Data/raw"
WORK = "/content/tcia"
os.makedirs(WORK, exist_ok=True)

TCIA = "https://faspex.cancerimagingarchive.net/aspera/faspex/public/package?context="
FILES = "https://www.cancerimagingarchive.net/wp-content/uploads/"
DATASETS = {
    "rhuh": dict(folder="longitudinal_rhuh_gbm", expect="40 patients x 3 timepoints (720 NIfTI files)",
                 package=TCIA + "eyJyZXNvdXJjZSI6InBhY2thZ2VzIiwidHlwZSI6ImV4dGVybmFsX2Rvd25sb2FkX3BhY2thZ2UiLCJpZCI6IjY4NCIsInBhc3Njb2RlIjoiNjFiMWU3OWZkYTU4MmRhNTBjZmU1NDZmYjkyZDQ0MzI2NjZjODZjNCIsInBhY2thZ2VfaWQiOiI2ODQiLCJlbWFpbCI6ImhlbHBAY2FuY2VyaW1hZ2luZ2FyY2hpdmUubmV0In0=",
                 extras=["clinical_data_TCIA_RHUH-GBM.csv"]),
    "mu": dict(folder="longitudinal_mu_glioma_post", expect="203 patients, about 596 timepoints (about 2,978 NIfTI files)",
               package=TCIA + "eyJyZXNvdXJjZSI6InBhY2thZ2VzIiwidHlwZSI6ImV4dGVybmFsX2Rvd25sb2FkX3BhY2thZ2UiLCJpZCI6IjEwMzAiLCJwYXNzY29kZSI6ImYyMGMxZDY1ODgwOWM1MWJhNWE3NGIwNTY0NGE4YWNmYzVlYTBmZjUiLCJwYWNrYWdlX2lkIjoiMTAzMCIsImVtYWlsIjoiaGVscEBjYW5jZXJpbWFnaW5nYXJjaGl2ZS5uZXQifQ==",
               extras=["MU-Glioma-Post_ClinicalData-July2025.xlsx", "MU-Glioma-Post_Segmentation_Volumes.xlsx", "MR_Scanner_data.xlsx"]),
    "africa": dict(folder="external_brats_africa", expect="146 patients (about 730 NIfTI files)",
                   package=TCIA + "eyJyZXNvdXJjZSI6InBhY2thZ2VzIiwidHlwZSI6ImV4dGVybmFsX2Rvd25sb2FkX3BhY2thZ2UiLCJpZCI6Ijk0OCIsInBhc3Njb2RlIjoiOTg2MzVlMGRmNzc3NWQ0NWJmZTQ2NjlhYzQwNjNmYjcxMjU0MzI1NyIsInBhY2thZ2VfaWQiOiI5NDgiLCJlbWFpbCI6ImhlbHBAY2FuY2VyaW1hZ2luZ2FyY2hpdmUubmV0In0=",
                   extras=["BraTS-Africa_TCIA_datainfo_v2.xlsx"]),
    "utsw": dict(folder="external_utsw_glioma", expect="625 patients (6,349 NIfTI files)",
                 package=TCIA + "eyJyZXNvdXJjZSI6InBhY2thZ2VzIiwidHlwZSI6ImV4dGVybmFsX2Rvd25sb2FkX3BhY2thZ2UiLCJpZCI6IjEyNjUiLCJwYXNzY29kZSI6IjM0MzA0NTU2NTgxNDA0ZjYyZTU1ODcwNWI4NjYyZWFjYjlkZGZiNjUiLCJwYWNrYWdlX2lkIjoiMTI2NSIsImVtYWlsIjoiaGVscEBjYW5jZXJpbWFnaW5nYXJjaGl2ZS5uZXQifQ==",
                 extras=["UTSW_Glioma_Metadata-2-1.tsv"]),
}
unknown = [c for c in COHORTS if c not in DATASETS]
assert not unknown, f"Unknown dataset(s): {unknown}. Use: {list(DATASETS)}"
!df -h /content | tail -1

In [ ]:
%%bash
# 2. Install Ruby 3.3 (via RVM) and IBM ascli, as in TCIA's official notebook. About 10-15 minutes.
set -e
if [ -f /etc/profile.d/rvm.sh ] && bash -lc "source /etc/profile.d/rvm.sh && rvm use 3.3.4 >/dev/null && command -v ascli" >/dev/null 2>&1; then
  echo "ascli already installed"; exit 0
fi
apt-get -qq update && apt-get -qq install -y gnupg2 curl build-essential > /dev/null
gpg2 --keyserver hkp://keyserver.ubuntu.com --recv-keys 409B6B1796C275462A1703113804BB82D39DC0E3 7D2BAF1CF37B13E2069D6956105BD0E739499BDB \
  || (curl -sSL https://rvm.io/mpapis.asc | gpg2 --import - && curl -sSL https://rvm.io/pkuczynski.asc | gpg2 --import -)
curl -sSL https://get.rvm.io | bash -s stable > /dev/null
source /etc/profile.d/rvm.sh
rvm install 3.3.4 > /dev/null
rvm use 3.3.4 --default
gem install aspera-cli --no-document > /dev/null
ascli config ascp install
ascli --version

In [ ]:
# 3. Download each dataset, check it, pack it into one .tar and copy it to Drive
import collections, shlex, shutil, subprocess, urllib.request
from IPython import get_ipython

def sh(cmd):
    """Run a command with Ruby/ascli available, showing its progress in the cell."""
    ip = get_ipython()
    ip.system("bash -c " + shlex.quote("source /etc/profile.d/rvm.sh && rvm use 3.3.4 >/dev/null && " + cmd))
    if ip.user_ns.get("_exit_code", 0) != 0:
        raise RuntimeError(f"command failed: {cmd[:80]}")

for key in COHORTS:
    info = DATASETS[key]
    local, dest = f"{WORK}/{info['folder']}", f"{DRIVE_RAW}/{info['folder']}"
    archive = f"{dest}/{info['folder']}.tar"
    os.makedirs(dest, exist_ok=True)
    print(f"\n===== {key} -> {dest}")
    for name in info["extras"]:                       # small clinical / metadata files, straight to Drive
        urllib.request.urlretrieve(FILES + name, f"{dest}/{name}")
        print("  saved", name)
    if os.path.exists(archive):
        print("  already on Drive, skipping download:", os.path.basename(archive))
        continue

    os.makedirs(local, exist_ok=True)
    try:
        sh(f"cd '{local}' && ascli faspex5 packages receive --url='{info['package']}'")
    except RuntimeError as e:
        print(f"  {key} FAILED ({e}). Re-run this cell to resume, or download it in the browser on your PC.")
        continue

    kinds = collections.Counter()
    for _, _, files in os.walk(local):
        for f in files:
            kinds[".nii.gz" if f.endswith(".nii.gz") else os.path.splitext(f)[1] or "(none)"] += 1
    size = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(local) for f in fs) / 1e9
    print(f"  downloaded {size:.1f} GB: {dict(kinds)}")
    print(f"  expected:   {info['expect']}")

    tmp = f"{WORK}/{info['folder']}.tar"             # pack on the Colab disk, then one big copy to Drive
    subprocess.run(["tar", "-cf", tmp, "-C", WORK, info["folder"]], check=True)
    shutil.copyfile(tmp, archive + ".part")
    os.replace(archive + ".part", archive)
    ok = os.path.getsize(archive) == os.path.getsize(tmp)
    print(f"  on Drive: {os.path.basename(archive)} {os.path.getsize(archive) / 1e9:.2f} GB {'OK' if ok else 'SIZE MISMATCH'}")
    shutil.rmtree(local); os.remove(tmp)             # free the Colab disk for the next dataset

In [ ]:
# 4. Finish uploading to Drive (large files keep syncing after the copy returns)
drive.flush_and_unmount()
print("Done. Check MyDrive/BrainMRI_Data/raw/ on drive.google.com: one .tar per dataset + its clinical files.")

## Using the data in Colab

Every dataset is stored the same way (one `.tar` per dataset). At the start of a session copy it to the Colab disk and
extract it there. Don't extract on Drive: reading thousands of small files through Drive is very slow.

```python
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/data && tar -xf /content/drive/MyDrive/BrainMRI_Data/raw/longitudinal_rhuh_gbm/longitudinal_rhuh_gbm.tar -C /content/data
```

**Citation:** all four collections are CC BY 4.0; cite each collection's DOI (on its TCIA page) and the TCIA paper (Clark et al., 2013).